In [ ]:
!pip install oxidd==0.11.2

In [ ]:
import sys
print(sys.version)

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [ ]:
pokec_small_nodes_path = "/content/Graphs/pokec-small_nodes.csv"
pokec_small_edges_path = "/content/Graphs/pokec-small_edges.csv"
pokec_medium_nodes_path = "/content/Graphs/pokec-medium_nodes.csv"
pokec_medium_edges_path= "/content/Graphs/pokec-medium_edges.csv"
pokec_large_nodes_path = "/content/Graphs/pokec_nodes.csv"
pokec_large_edges_path = "/content/Graphs/pokec_edges.csv"

congress_nodes_path = "/content/Graphs/congress_nodes.csv"
congress_edges_path = "/content/Graphs/congress_edges.csv"


In [ ]:
import pandas as pd

def get_nodes_pokec (nodes, path):
  df = pd.read_csv(path, sep='\t')

  for index, row in df.iterrows():
      id = row['id']
      label = row['label']
      nodes[id] = label

  return nodes

def get_transitions_pokec (transitions, path):
  df = pd.read_csv(path, sep='\t')

  for index, row in df.iterrows():
      source = int(row['source'])
      target = int(row['target'])
      transitions.add((source, target))

  return transitions

In [ ]:
import math
from oxidd.bdd import BDDManager

nodes = {}
transitions = set()

# nodes = get_nodes_pokec(nodes, pokec_small_nodes_path)
# transitions = get_transitions_pokec(transitions, pokec_small_edges_path)

nodes = get_nodes_pokec(nodes, pokec_large_nodes_path)
transitions = get_transitions_pokec(transitions, pokec_large_edges_path)

S = len(nodes)
l = math.ceil(math.log2(S))
print(f"Size of S: {S}")
print(f"Size of l: {l}")

manager = BDDManager(100_000_000, 1_000_000, 1)
x = [manager.var(i) for i in manager.add_vars(l)]

Size of S: 1632803
Size of l: 21


In [ ]:
def Create_Label_1_Check_BDD ():
  BDD = manager.false()
  for id, value in nodes.items():
    if value == 0:
      continue;

    encoding = f"{id:0{l}b}"

    term = manager.true()
    for i in range(l):
      if encoding[i] == '1':
        term &= x[i]
      else:
        term &= ~x[i]

    BDD = BDD | term

  return BDD

def Create_Label_0_Check_BDD ():
  BDD = manager.false()
  for id, value in nodes.items():
    if value == 1:
      continue;

    encoding = f"{id:0{l}b}"

    term = manager.true()
    for i in range(l):
      if encoding[i] == '1':
        term &= x[i]
      else:
        term &= ~x[i]

    BDD = BDD | term

  return BDD

def Create_Id_Check_BDD ():
  BDD = manager.false()
  for id, value in nodes.items():
    encoding = f"{id:0{l}b}"

    term = manager.true()
    for i in range(l):
      if encoding[i] == '1':
        term &= x[i]
      else:
        term &= ~x[i]

    BDD = BDD | term

  return BDD

def Create_Transition_Check_BDD ():
  BDD = manager.false()

  for source, target in transitions:
    encoding_target = f"{source:0{l}b}"
    encoding_source = f"{target:0{l}b}"
    encoding = encoding_source + encoding_target

    term = manager.true()
    for i in range(l):
      if encoding[i] == '1':
        term &= x[i]
      else:
        term &= ~x[i]

    BDD = BDD | term

  return BDD

def check_node_id(manager, bdd, x, node_id, l):
    encoding = f"{node_id:0{l}b}"

    term = manager.true()
    for i in range(l):
        if encoding[i] == '1':
            term &= x[i]
        else:
            term &= ~x[i]

    return (bdd & term) != manager.false()

def check_transition(manager, bdd, x, source, target, l):
    encoding_target = f"{source:0{l}b}"
    encoding_source = f"{target:0{l}b}"
    encoding = encoding_source + encoding_target

    minterm = manager.true()
    for i in range(l):
        if encoding[i] == '1':
            minterm &= x[i]
        else:
            minterm &= ~x[i]
    return (bdd & minterm) != manager.false()

BDD_0 = Create_Label_0_Check_BDD()
BDD_1 = Create_Label_1_Check_BDD()
BDD_id = Create_Id_Check_BDD()
BDD_transition = Create_Transition_Check_BDD()

552525
1080278
1632803
1519452


In [ ]:
def X_0 (id):
  return check_node_id(manager, BDD_0, x, id, l)

def X_1 (id):
  return check_node_id(manager, BDD_1, x, id, l)

def T (source, target):
  return check_transition(manager, BDD_id, x, source, target, l)

def nodesReachableFrom1(unvisited_nodes):
  count = 0
  stack = [1]
  unvisited_nodes.remove(1)

  while (len(stack) > 0):
    current_node = stack.pop()

    reachable_nodes = []
    for node in unvisited_nodes:
      if T(current_node, node):
        reachable_nodes.append(node)

    for node in reachable_nodes:
        stack.append(node)
        unvisited_nodes.remove(node)
        count += 1

    return count

def nodesReachableFrom1With01Path(unvisited_nodes):
  count = 0
  stack = [1]
  unvisited_nodes.remove(1)

  while (len(stack) > 0):
    current_node = stack.pop()

    reachable_nodes = []
    for node in unvisited_nodes:
      if T(current_node, node) and X_0(current_node) != X_0(node):
        reachable_nodes.append(node)

    for node in reachable_nodes:
        stack.append(node)
        unvisited_nodes.remove(node)
        count += 1

    return count

unvisited_nodes = set(nodes.keys())

print("Nodes reachable:")
print(nodesReachableFrom1(unvisited_nodes))

unvisited_nodes = set(nodes.keys())

print("Nodes reachable by 0/1 path")
print(nodesReachableFrom1With01Path(unvisited_nodes))

def findPath (i, current_node, current_path, unvisited_nodes):
  longest_path = current_path

  for node in unvisited_nodes:
    if T(current_node, node) == False or node in current_path:
      continue

    temp_unvisited = unvisited_nodes.copy()
    temp_unvisited.remove(node)

    temp_path = current_path.copy()
    temp_path.append(node)

    path = []

    #Even
    if i%2 == 0 and X_1(node):
      path = findPath(i+1, node, temp_path, temp_unvisited)

    #Odd
    if i%2 != 0 and X_0(node):
      path = findPath(i+1, node, temp_path, temp_unvisited)

    if len(path) > len(longest_path):
      longest_path = path

  return longest_path

# unvisited = set(nodes.keys())
# start_node = 1
# print(findPath(0, 1, [1], unvisited))

Nodes reachable:
1632802
Nodes reachable by 0/1 path
1632803
552525


In [ ]:
# print("Transition check:")
# print(check_transition(manager, BDD_id, x, 1, 13, l))
# print(check_transition(manager, BDD_id, x, 1, 11, l))
# print(check_transition(manager, BDD_id, x, 1, 6, l))
# print(check_transition(manager, BDD_id, x, 1, 3, l))
# print(check_transition(manager, BDD_id, x, 1, 4, l))
# print(check_transition(manager, BDD_id, x, 1, 5, l))
# print(check_transition(manager, BDD_id, x, 1, 15, l))
# print(check_transition(manager, BDD_id, x, 1, 2, l))
# print(check_transition(manager, BDD_id, x, 2, 1, l))

# print("Id check:")
# print(check_node_id(manager, BDD_id, x, 1, l))
# print(check_node_id(manager, BDD_id, x, 2, l))
# print(check_node_id(manager, BDD_id, x, 16, l))
# print(check_node_id(manager, BDD_id, x, 3, l))
# print(check_node_id(manager, BDD_id, x, 4, l))
# print(check_node_id(manager, BDD_id, x, 17, l))
# print(check_node_id(manager, BDD_id, x, 5, l))

# print("Label 0 check:")
# print(check_node_id(manager, BDD_0, x, 1, l))
# print(check_node_id(manager, BDD_0, x, 2, l))
# print(check_node_id(manager, BDD_0, x, 16, l))
# print(check_node_id(manager, BDD_0, x, 3, l))
# print(check_node_id(manager, BDD_0, x, 4, l))
# print(check_node_id(manager, BDD_0, x, 17, l))
# print(check_node_id(manager, BDD_0, x, 5, l))

# print("Label 1 check:")
# print(check_node_id(manager, BDD_1, x, 1, l))
# print(check_node_id(manager, BDD_1, x, 2, l))
# print(check_node_id(manager, BDD_1, x, 16, l))
# print(check_node_id(manager, BDD_1, x, 3, l))
# print(check_node_id(manager, BDD_1, x, 4, l))
# print(check_node_id(manager, BDD_1, x, 17, l))
# print(check_node_id(manager, BDD_1, x, 5, l))